In [162]:
import shap
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from shap import initjs
import math

# Übung 6: Shapley Values

### Fragen zum Auffrischen vorab
1. Was ist die Motivation hinter Shapley Values?
2. Wie wird die Shapley Idee für ML realisiert?
3. Was ist ein praktisches Problem von Shapley values und wie wird es gelöst=
4. Ja oder Nein: Shapley values und SHAP values sind untterschiedliche Namen für dasselbe Konzept.
5. Was sind SHAP value functions $v$ im Gegensatz zu SHAP values $\phi$?
## a) Exakte Implementierung

In [163]:
def payoff(team):
    team = set(team)
    t = 't' in team
    s = 's' in team
    m = 'm' in team
    j = 'j' in team
    l = 'l' in team
    reward = 10*t + 10*m + 2*j + 20 * (t and m) + 20 * (t and m and s) - 30 * ((t or m or s) and j)
    return reward

In [198]:
payoff(['t', 'j', 's'])

-18

In [164]:
population = ['t', 'm', 's', 'j', 'l']

In [199]:
def all_unique_subsets(population):
    population = list(population)
    if len(population) == 0:
        return [[]]
    else:
        subsets = all_unique_subsets(population[1:])
        subsets_with = []
        for s in subsets:
            s_with = list(s)
            s_with.append(population[0])
            subsets_with.append(s_with)
        subsets = subsets + subsets_with
        return subsets

def shapley(member, population, vfunc):
    remainder = [ind for ind in population if member != ind]
    all_sets = all_unique_subsets(remainder)
    val = 0
    N = len(population)
    for s in all_sets:
        S = len(s)
        diff = vfunc(s + [member]) - vfunc(s)
        factor = math.factorial(S) * math.factorial(N-S-1) / math.factorial(N)
        val += factor * diff
    return val

In [203]:
shapley('s', ['t', 'm', 's', 'j', 'l'], payoff)

4.166666666666666

## b) Annährung durch Permutationen

In [167]:
def shapley_perm(member, population, vfunc, *args, its=100):
    vals = []
    for ii in range(its):
        perm = np.random.permutation(population)
        member_ix = np.where(perm == member)
        s = perm[:member_ix[0][0]].tolist()
        diff = vfunc(s + [member], *args) - vfunc(s, *args)
        vals.append(diff)
    val = sum(vals)/len(vals)
    return val

In [188]:
shapley_perm('t', ['t', 'm', 's', 'j', 'l'], payoff, its=100000)

24.1732

## c) Axiome

### i) Symmetry Check

In [169]:
def symmetry_check(j, k, population, vfunc, shapley_func):
    remainder = set(population) - set([j, k])
    all_S = all_unique_subsets(remainder)
    surpluss_j = []
    surpluss_k = []
    for S in all_S:
        surplus_j = vfunc(S + [j]) - vfunc(S)
        surplus_k = vfunc(S + [k]) - vfunc(S)
        surpluss_j.append(surplus_j)
        surpluss_k.append(surplus_k)
    surpluss_j, surpluss_k = np.array(surpluss_j), np.array(surpluss_k)
    equal_surplus = np.all(surpluss_j == surpluss_k)
    if equal_surplus:
        print('equal surplus')
        val_j = shapley_func(j, population, vfunc)
        val_k = shapley_func(k, population, vfunc)
        return val_j == val_k
    else:
        return True


In [170]:
symmetry_check('m', 't', population, payoff, shapley)

equal surplus


True

### ii) Dummy property check

In [171]:
def dummy_check(j, population, vfunc, shapley_func):
    remainder = set(population) - set([j])
    all_S = all_unique_subsets(remainder)
    surpluss_j = []
    for S in all_S:
        surplus_j = vfunc(S + [j]) - vfunc(S)
        surpluss_j.append(surplus_j)
    has_contribution = np.sum(np.abs(surpluss_j)) > 0
    if has_contribution:
        print('has contribution')
        val_j = shapley_func(j, population, vfunc)
        return val_j > 0
    else:
        return True

In [172]:
dummy_check('l', population, payoff, shapley)

True

### iii) Additivity check

In [173]:
def additivity_check(population, vfunc1, vfunc2, shapley_func):
    combined = lambda x : vfunc1(x) + vfunc2(x)
    vals1 = np.array([shapley_func(j, population, vfunc1) for j in population])
    vals2 = np.array([shapley_func(j, population, vfunc2) for j in population])
    vals_comb = np.array([shapley_func(j, population, combined) for j in population])
    vals_additive = vals1 + vals2
    return np.all(vals_comb == vals_additive)


In [174]:
payoff2 = payoff

additivity_check(population, payoff, payoff2, shapley)

np.True_

### iv): Efficiency check

In [175]:
def efficiency_check(population, vfunc, shapley_func):
    payoff_total = vfunc(population)
    shapley_vals = [shapley_func(j, population, vfunc) for j in population]
    total_shapley_vals = np.sum(shapley_vals)
    pt, st = round(payoff_total, 5), round(total_shapley_vals, 5)
    return pt == st

In [176]:
efficiency_check(population, payoff, shapley)

np.True_

##  Hausaufgabe 2: SHAP

### a) Laden Sie den FIFA Datensatz.
Laden Sie den FIFA Datensatz in eine Datentabelle (z.B. mit Pandas) und sagen Sie die "Man of the Match" Wahrscheinlichkeit mit einem Randomforest voraus.

_Hinweis:_ Transformieren Sie die Zielvariable "Man of the Match" in ein binäres Format, das für eine Vorhersage geeignet ist, und wählen Sie nur ganzzahlige Merkmale, da viele der Float-Variablen im Datensatz fehlende Werte enthalten. Vergessen Sie nicht, die Daten in einen Trainings- und Testsatz (70:30, random_state=42) aufzuteilen, bevor Sie den Randomforest trainieren!

### b) Berechnen Sie die SHAP values.
Berechnen Sie mittels der `shap` Bibliothek die SHAP values auf dem Testdatensatz. Nutzen Sie zwei unterschiedliche Visualisierungsmethoden der `shap.plots` Klasse, um die erste Instanz des Testsatzes zu erklären. Beschreiben Sie die gelieferten Erklärungen kurz und vergleichen Sie die beiden Darstellungen vor dem Hintergrund der Inhalte aus den Vorlesungen zu Visualisierungen!

_Hinweis:_ Achten Sie auf die Form des Arrays der shap values, falls Sie einen Fehler erhalten. Wir sind an der Erklärung für die Wahrscheinlichkeit, dass der "Man of the Match" im Team ist (1), interessiert!

### c) Von lokalen Erklärungen zu globalen Merkmalseffekten
Welche Methoden in `shap.plots` sind dafür geeignet, die Relevanz einzelner Merkmale (feature importance) mit den Ausprägungen der Merkmale (Feature Effects) zu kombinieren? Begründen Sie Ihre Wahl kurz und zeigen Sie den Plot!

### d) SHAP Heatmap
Lassen Sie sich die heatmap für den Testdatensatz ausgeben und beschreiben Sie die Visualisierung kurz.

### e) Detaillierte Analyse von Interaktionen
Eventuell interagieren einzelne Variablen miteinander - z.B. könnte man das bei "Goal Scored" und "Attempts" erwarten. Implementieren Sie eine ähnliche Visualisierung zur Interaktion der beiden Variablen, wie sie in den Vorlesungsfolien dargestellt ist.